In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

!pip install --upgrade transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.2/517.2 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 34.9 MB/s eta 0:00:00


In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Count: {torch.cuda.device_count()}")
print(f"Current Device: {torch.cuda.current_device()}")

GPU: NVIDIA L4
GPU Count: 1
Current Device: 0


In [ ]:


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import pandas as pd
import random

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/content/drive/MyDrive/llmFinal/qwen2-1.5b-finetuned"
TARGET_SIZE = 5000

DATASET_PATHS = [
    "airport_ticket.csv",
    "fastfood.csv",
    "finance.csv",
    "media.csv",
    "software.csv",
]

SYSTEM_MESSAGE = """You are a professional customer service assistant.
IMPORTANT RULES:
- Respond ONLY in English using ASCII characters
- Be concise: maximum 2-3 sentences per response
- Never repeat yourself or generate the same phrase multiple times
- Stay focused on answering the customer's question directly"""

print("="*70)
print("STEP 1: Data Cleaning")
print("="*70)

def aggressive_clean(csv_path):
    df = pd.read_csv(csv_path)
    original_size = len(df)

    df = df.dropna(subset=['utterance', 'conversationId', 'authorRole'])
    df['utterance'] = df['utterance'].astype(str).str.strip()
    df = df[df['utterance'] != '']
    df = df[df['utterance'].str.lower() != 'nan']

    def clean_text(text):
        cleaned = ''.join(char for char in text if 32 <= ord(char) < 127)
        return cleaned.strip()

    df['utterance'] = df['utterance'].apply(clean_text)
    df = df[df['utterance'].str.len() > 0]
    df = df[(df['utterance'].str.len() >= 2) & (df['utterance'].str.len() <= 250)]
    df = df.drop_duplicates(subset=['conversationId', 'turnNumber'])
    df = df.sort_values(['conversationId', 'turnNumber'])

    conv_lengths = df.groupby('conversationId').size()
    valid_convs = conv_lengths[(conv_lengths >= 4) & (conv_lengths <= 25)].index
    df = df[df['conversationId'].isin(valid_convs)]

    def has_both_roles(group):
        roles = set(group['authorRole'])
        return 'customer' in roles and 'agent' in roles

    valid_convs = df.groupby('conversationId').filter(has_both_roles)['conversationId'].unique()
    df = df[df['conversationId'].isin(valid_convs)]

    cleaned_pct = len(df) / original_size * 100 if original_size > 0 else 0
    print(f"  {csv_path.split('/')[-1]}: {original_size} → {len(df)} ({cleaned_pct:.1f}%)")

    return df

cleaned_dfs = []
for path in DATASET_PATHS:
    try:
        cleaned_df = aggressive_clean(path)
        cleaned_dfs.append(cleaned_df)
    except Exception as e:
        print(f" Error: {e}")

all_data = pd.concat(cleaned_dfs, ignore_index=True)
print(f"\nTotal: {len(all_data)} rows, {all_data['conversationId'].nunique()} conversations")

print("\n" + "="*70)
print("STEP 2: Processing Conversations")
print("="*70)

def process_to_conversations(df):
    df = df.sort_values(['conversationId', 'turnNumber'])
    conversations = []

    for conv_id, group in df.groupby('conversationId'):
        messages = []
        for _, row in group.iterrows():
            role = "user" if row['authorRole'] == 'customer' else "assistant"
            messages.append({"role": role, "content": row['utterance']})

        if len(messages) >= 4 and len(messages) <= 25:
            if messages[0]['role'] == 'assistant':
                messages = messages[1:]
            if messages[-1]['role'] == 'user':
                messages = messages[:-1]
            if len(messages) >= 4:
                conversations.append(messages)

    return conversations

conversations = process_to_conversations(all_data)
print(f"Processed: {len(conversations)} conversations")

random.seed(42)
if len(conversations) > TARGET_SIZE:
    conversations = random.sample(conversations, TARGET_SIZE)
    print(f"Sampled to: {len(conversations)}")

print("\n" + "="*70)
print("STEP 3: Loading Qwen 2.5-1.5B")
print("="*70)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model = get_peft_model(model, lora_config)
print(f"\nModel loaded!")
model.print_trainable_parameters()

print("\n" + "="*70)
print("STEP 4: Formatting Dataset")
print("="*70)

formatted_data = []
skipped = 0

for conversation in conversations:
    try:
        messages_with_system = [
            {"role": "system", "content": SYSTEM_MESSAGE}
        ] + conversation

        text = tokenizer.apply_chat_template(
            messages_with_system,
            tokenize=False,
            add_generation_prompt=False
        )

        if len(text) < 2048:
            formatted_data.append({"text": text})
        else:
            skipped += 1
    except:
        skipped += 1

print(f"Formatted: {len(formatted_data)}")
print(f"Skipped: {skipped}")

dataset = Dataset.from_list(formatted_data)
dataset = dataset.shuffle(seed=42)

print(f"\nSample:")
print(formatted_data[0]["text"][:500])
print("...")


print("\n" + "="*70)
print("STEP 5: Training Setup")
print("="*70)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    save_steps=500,
    logging_steps=50,
    save_total_limit=2,
    warmup_steps=100,
    bf16=True,
    max_grad_norm=0.3,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

print("\n" + "="*70)
print("STEP 6: Starting Training")
print("="*70)
print("="*70 + "\n")

trainer.train()

print("\n" + "="*70)
print("STEP 7: Saving")
print("="*70)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n Done! Saved to: {OUTPUT_DIR}")


print("\n" + "="*70)
print("STEP 8: Quick Test")
print("="*70)

test_messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": "I want to book a flight to Chicago"}
]

text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nResponse:")
print("="*70)
print(response)


STEP 1: Data Cleaning
  airport_ticket.csv: 237514 → 223708 (94.2%)
  fastfood.csv: 180159 → 156497 (86.9%)
  finance.csv: 157229 → 138627 (88.2%)
  media.csv: 485961 → 467345 (96.2%)
  software.csv: 86220 → 82547 (95.7%)

✅ Total: 1068724 rows, 69585 conversations

STEP 2: Processing Conversations
✅ Processed: 69555 conversations
✅ Sampled to: 5000

STEP 3: Loading Qwen 2.5-1.5B
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (9 outputs left out)

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



STEP 6: Starting Training
⏰ Estimated: 5-10 minutes on L4



Step,Training Loss
50,2.194100
100,1.197600
150,1.046400
200,0.966100
250,0.916800
300,0.888900
350,0.830600
400,0.818800
450,0.808200
500,0.786800



STEP 7: Saving

✅ Done! Saved to: /content/drive/MyDrive/llmFinal/qwen2-1.5b-finetuned

STEP 8: Quick Test

Response:
system
You are a professional customer service assistant.
IMPORTANT RULES:
- Respond ONLY in English using ASCII characters
- Be concise: maximum 2-3 sentences per response
- Never repeat yourself or generate the same phrase multiple times
- Stay focused on answering the customer's question directly
user
I want to book a flight to Chicago
assistant
Sure! I'll help you with that. Could please help me with your departure and arrival cities?
